# OmniForge Enterprise Quickstart
Auto-generated by OmniForge Enterprise (`domain=finance`).

In [ ]:
import omniforge
from omniforge.automl import AutoMLPipelineSearch
from omniforge.data import DataCleaner, DataQualityProfiler, SyntheticDataGenerator
from omniforge.eda import AutomatedEDA
from omniforge.evaluation import ModelCardGenerator

print('OmniForge', omniforge.__version__)
omniforge.set_global_seed(42)

In [ ]:
# Self-contained synthetic dataset (no external CSV needed, so the notebook always runs)
df_raw = SyntheticDataGenerator(random_state=42).generate_classification_dataset(
    n_samples=400, n_features=8, missing_rate=0.05
)
quality = DataQualityProfiler().profile(df_raw)
print('Quality Score:', quality.overall_score, quality.grade)
df_clean, clean_report = DataCleaner().fit_transform(df_raw)
print('Missing cells:', clean_report.missing_values_before, '->', clean_report.missing_values_after)
df_clean.head()

In [ ]:
eda_report = AutomatedEDA().run(df_clean, target_column='target', generate_figures=False)
print(eda_report.narrative)

In [ ]:
X = df_clean.drop(columns=['target'])
y = df_clean['target']
automl_res = AutoMLPipelineSearch(time_budget_seconds=60.0, tune_trials=4).fit(X, y)
print('Champion:', automl_res.best_model_name, automl_res.primary_metric, automl_res.best_score)
print('Lift over best baseline:', automl_res.lift_over_best_baseline)

In [ ]:
# `automl_res.metadata` is the search-level dict; the per-model ModelMetadata
# (carbon report, training rows, feature names) lives on the champion estimator.
champion_row = next(
    (row for row in automl_res.leaderboard if row.get('model_name') == automl_res.best_model_name),
    {},
)
card = ModelCardGenerator.build_model_card(
    automl_res.best_model.metadata_,
    evaluation_metrics={automl_res.primary_metric: champion_row.get('cv_score', automl_res.best_score)},
    baseline_comparison={'lift_over_best_baseline': automl_res.lift_over_best_baseline},
)
print(card.to_markdown())